# Lesson 1.4 — Business Transactions

## Learning Objectives
- Define what a Business Transaction (BT) is in AppDynamics
- Understand how AppDynamics auto-discovers Business Transactions
- Explain BT naming conventions, grouping, and scoping
- List Business Transactions via the REST API

**Reference**: [Business Transactions](https://help.splunk.com/en/appdynamics-saas/application-performance-monitoring/26.6.0/business-transactions)

## 1. What is a Business Transaction?

A **Business Transaction (BT)** is the unit of work that AppDynamics tracks end-to-end.

Think of it this way:
- A user clicks **"Add to Cart"** → AppDynamics starts a BT called `/cart/add`
- The request flows through your WebFrontend → OrderService → InventoryAPI → Redis cache
- AppDynamics tracks the **entire journey**, recording how long each hop took
- When the response returns to the user, the BT completes

**Key insight**: BTs connect **technical performance** (response time, errors) to **business actions** (checkout, search, login). This is what makes AppDynamics different from simple infrastructure monitoring.

### What AppDynamics measures per BT:
| Metric | Description |
|--------|-------------|
| **Calls per Minute** | How often this transaction is being invoked |
| **Average Response Time** | Mean time from start to completion (ms) |
| **Error Rate** | % of calls that result in an error |
| **Slow Threshold** | How long before a call is labeled "Slow" |
| **Very Slow Threshold** | How long before labeled "Very Slow" |
| **Stall Count** | Calls that never completed within the stall threshold |

In [ ]:
# Simulate Business Transaction performance data
import random, statistics
from collections import defaultdict

random.seed(7)

# Simulate 1 minute of BT calls
business_transactions = {
    "/checkout": {"calls": 42, "times": [random.gauss(320, 100) for _ in range(42)]},
    "/search":   {"calls": 210, "times": [random.gauss(95, 30) for _ in range(210)]},
    "/cart/add": {"calls": 87, "times": [random.gauss(150, 50) for _ in range(87)]},
    "/login":    {"calls": 33, "times": [random.gauss(200, 200) for _ in range(33)]},  # noisy
}

SLOW_THRESHOLD      = 500   # ms
VERY_SLOW_THRESHOLD = 2000  # ms

print(f"{'Business Transaction':<20} {'Calls/Min':>9} {'Avg RT (ms)':>11} {'P95 (ms)':>9} {'Slow%':>7} {'VSlw%':>7}")
print("-" * 70)

for bt_name, data in business_transactions.items():
    times = data["times"]
    avg_rt = statistics.mean(times)
    p95    = sorted(times)[int(len(times) * 0.95)]
    slow   = sum(1 for t in times if SLOW_THRESHOLD < t <= VERY_SLOW_THRESHOLD)
    vslow  = sum(1 for t in times if t > VERY_SLOW_THRESHOLD)
    print(f"{bt_name:<20} {data['calls']:>9} {avg_rt:>11.1f} {p95:>9.1f} "
          f"{slow/len(times)*100:>7.1f} {vslow/len(times)*100:>7.1f}")

## 2. How AppDynamics Discovers Business Transactions

AppDynamics uses **Entry Points** to define where a BT starts. It auto-detects common patterns:

| Entry Point Type | How BT Name is Derived | Example |
|-----------------|-------------------------|----------|
| **Servlet** (Java) | HTTP URI path | `/checkout` → BT name = `/checkout` |
| **ASP.NET** (.NET) | Request path | `/api/orders` → BT name = `/api/orders` |
| **Node.js HTTP** | Express route | `GET /users/:id` → BT name = `/users/{dynamic}` |
| **POJO** | Class + method name | `OrderService.processOrder()` |
| **Message Queue** | Queue name + operation | `kafka.orders.consumed` |
| **Spring** | Controller method | `@RequestMapping("/cart")` → `/cart` |

### Discovery Rules
AppDynamics uses **Transaction Detection Rules** to customize BT discovery:
- **Automatic rules**: Default rules that work out of the box for common frameworks
- **Custom rules**: You define which URI segments are significant vs. dynamic
  - Without customization: `/users/123`, `/users/456` → two BTs (noisy!)
  - With a custom rule: `/users/*` → one BT called `/users/{userId}`

> **Gotcha**: Too many BT entries ("BT explosion") wastes license and makes dashboards unreadable. Use custom rules to group dynamic segments.

In [ ]:
# Demonstrate BT grouping strategy

# Problem: Without grouping, each unique user ID creates a separate BT
raw_requests = [
    "/users/1001/orders",
    "/users/1002/orders",
    "/users/1003/orders",
    "/products/PROD-001",
    "/products/PROD-002",
    "/checkout",
    "/checkout",
]

# Without grouping rules
without_grouping = set(raw_requests)
print("WITHOUT grouping rules:")
for bt in sorted(without_grouping):
    print(f"  BT: {bt}")
print(f"  Total BTs: {len(without_grouping)}")
print()

# With grouping rules (custom transaction detection)
import re

def apply_bt_rule(uri: str) -> str:
    """Applies grouping rules to normalize dynamic URI segments."""
    uri = re.sub(r'/users/\d+', '/users/{userId}', uri)
    uri = re.sub(r'/products/[A-Z0-9-]+', '/products/{productId}', uri)
    return uri

with_grouping = set(apply_bt_rule(r) for r in raw_requests)
print("WITH grouping rules:")
for bt in sorted(with_grouping):
    print(f"  BT: {bt}")
print(f"  Total BTs: {len(with_grouping)}")
print()
print(f"BT count reduction: {len(without_grouping)} → {len(with_grouping)}")

## 3. BT Scoping and Exclusion

Not every request should be a tracked BT. AppDynamics provides controls:

**Exclude patterns** — common things to exclude:
- Health check endpoints: `/health`, `/ping`, `/actuator/health`
- Static asset requests: `/static/*`, `*.css`, `*.js`, `*.png`
- Internal service-to-service polling

**Scope** — BTs can be scoped to specific tiers:
- A BT starting in `WebFrontend` can be tracked as it calls `OrderService` and `InventoryAPI`
- Each tier segment shows up in the **Transaction Snapshot** (call graph)

### Transaction Snapshots

When a BT is slow, AppDynamics automatically captures a **Transaction Snapshot**:
- Full call graph showing every method called
- Time spent in each method and exit call
- SQL queries executed with bind parameters
- Stack traces for errors

This is the **most powerful debugging feature** in AppDynamics — it shows exactly where time was spent.

In [ ]:
# REST API: List Business Transactions for an application

import os, requests

CONTROLLER_URL = os.getenv("APPD_CONTROLLER_URL", "https://your-account.saas.appdynamics.com")
ACCOUNT_NAME   = os.getenv("APPD_ACCOUNT",        "your-account")
USERNAME       = os.getenv("APPD_USER",            "admin")
PASSWORD       = os.getenv("APPD_PASSWORD",        "")

def appd_get(endpoint):
    url  = f"{CONTROLLER_URL}/controller/rest/{endpoint}"
    resp = requests.get(url, auth=(f"{USERNAME}@{ACCOUNT_NAME}", PASSWORD),
                        params={"output": "JSON"}, timeout=30)
    resp.raise_for_status()
    return resp.json()

APP_ID = int(os.getenv("APPD_APP_ID", "0")) or None

if APP_ID:
    try:
        bts = appd_get(f"applications/{APP_ID}/business-transactions")
        print(f"Business Transactions in App ID {APP_ID} ({len(bts)} found):")
        for bt in bts[:15]:  # show first 15
            print(f"  ID={bt['id']:5d} | Tier={bt.get('tierName','?'):<20} | BT={bt['name']}")
        if len(bts) > 15:
            print(f"  ... and {len(bts) - 15} more")
    except Exception as e:
        print(f"Error: {e}")
else:
    print("APP_ID not set. Sample output:")
    sample_bts = [
        {"id": 201, "tierName": "WebFrontend",  "name": "/checkout"},
        {"id": 202, "tierName": "WebFrontend",  "name": "/search"},
        {"id": 203, "tierName": "WebFrontend",  "name": "/cart/add"},
        {"id": 204, "tierName": "OrderService", "name": "OrderService.processOrder"},
    ]
    for bt in sample_bts:
        print(f"  ID={bt['id']:5d} | Tier={bt['tierName']:<20} | BT={bt['name']}")

In [ ]:
# EXERCISE 1.4: Design transaction detection rules
#
# You're onboarding the 'StoreFront' application which has these raw URIs:
# /api/v1/users/1001/cart
# /api/v1/users/1002/cart
# /api/v1/products/SKU-ABC123
# /api/v1/products/SKU-XYZ789
# /api/v1/orders
# /api/v1/orders/ORD-001/status
# /api/v1/orders/ORD-002/status
# /health
# /metrics
#
# TASK: Write regex rules to normalize these into clean BT names
# TASK: Add a rule to exclude /health and /metrics

import re

raw_uris = [
    "/api/v1/users/1001/cart",
    "/api/v1/users/1002/cart",
    "/api/v1/products/SKU-ABC123",
    "/api/v1/products/SKU-XYZ789",
    "/api/v1/orders",
    "/api/v1/orders/ORD-001/status",
    "/api/v1/orders/ORD-002/status",
    "/health",
    "/metrics",
]

EXCLUDE_PATTERNS = [
    # TODO: Add regex patterns for URIs to exclude
]

GROUPING_RULES = [
    # TODO: (pattern_to_match, replacement) tuples
    # e.g., (r'/users/\d+', '/users/{userId}')
]

def process_uri(uri):
    for excl in EXCLUDE_PATTERNS:
        if re.match(excl, uri):
            return None  # excluded
    for pattern, replacement in GROUPING_RULES:
        uri = re.sub(pattern, replacement, uri)
    return uri

print("BT name results:")
bt_set = set()
for uri in raw_uris:
    result = process_uri(uri)
    if result:
        bt_set.add(result)
        print(f"  INCLUDED: {uri!s:<40} → {result}")
    else:
        print(f"  EXCLUDED: {uri}")
print(f"\nFinal BT count: {len(bt_set)}")

## Summary

- A **Business Transaction** is AppDynamics' unit of work — it tracks a user action end-to-end across all tiers
- AppDynamics **auto-discovers** BTs from entry points (Servlet, HTTP, MQ, POJO) based on the framework
- **Transaction Detection Rules** group dynamic URI segments to prevent BT explosion
- **Transaction Snapshots** capture full call graphs for slow/errored transactions
- BTs are the foundation for health rules, dashboards, and ADQL analytics — everything in AppDynamics revolves around BTs

---

**Phase 1 Complete!** Proceed to [Phase 2 — Monitoring Core](../phase-2-monitoring-core/lesson-2.1-metrics-and-metric-browser.ipynb)